<a href="https://colab.research.google.com/github/asknern/bist100-analiz/blob/main/bist100_portfolio_optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install yfinance PyPortfolioOpt

In [ ]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-darkgrid')

hisseler = ["THYAO.IS", "GARAN.IS", "EREGL.IS", "SASA.IS", "AKBNK.IS",
    "TUPRS.IS", "KCHOL.IS", "SAHOL.IS", "SISE.IS", "BIMAS.IS",
    "FROTO.IS", "ASELS.IS", "ENKAI.IS", "YKBNK.IS", "TOASO.IS"]

print("Borsa verileri çekiliyor, lütfen bekleyin...")

veriler = yf.download(hisseler, period="2y")['Close']
veriler = veriler.ffill()

plt.figure(figsize=(12, 6))
for hisse in hisseler:
    plt.plot(veriler.index, veriler[hisse], label=hisse.replace('.IS', ''))

plt.title("Seçili BIST 100 Hisseleri - Son 2 Yıllık Fiyat Hareketleri")
plt.xlabel("Tarih")
plt.ylabel("Fiyat (TL)")
plt.legend()
plt.show()

In [ ]:
from pypfopt import expected_returns, risk_models
from pypfopt.efficient_frontier import EfficientFrontier

print("Optimizasyon Algoritması Çalışıyor...\n")

mu = expected_returns.mean_historical_return(veriler)
S = risk_models.sample_cov(veriler)

ef = EfficientFrontier(mu, S)
weights = ef.max_sharpe()
cleaned_weights = ef.clean_weights()

print("🤖 MODELİN ÖNERDİĞİ PORTFÖY DAĞILIMI:")
print("-" * 40)
for hisse, oran in cleaned_weights.items():
    if oran > 0:
        print(f"👉 {hisse.replace('.IS', '')}: %{oran*100:.1f}")

print("\n📈 BU PORTFÖYÜN BEKLENEN PERFORMANSI:")
print("-" * 40)
ef.portfolio_performance(verbose=True)

labels = [hisse.replace('.IS', '') for hisse, oran in cleaned_weights.items() if oran > 0]
sizes = [oran for oran in cleaned_weights.values() if oran > 0]

plt.figure(figsize=(8, 8))
plt.pie(sizes, labels=labels, autopct='%1.1f%%', startangle=140,
        colors=['#4C72B0', '#55A868', '#C44E52', '#8172B3', '#CCB974'],
        textprops={'fontsize': 12, 'weight': 'bold'})
plt.title("Yapay Zeka Destekli Optimum Portföy Dağılımı (Maksimum Sharpe)", fontsize=14, fontweight='bold')
plt.show()

In [ ]:
from sklearn.ensemble import RandomForestRegressor
import numpy as np
import pandas as pd
from pypfopt import risk_models
from pypfopt.efficient_frontier import EfficientFrontier
from pypfopt import plotting
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')

print("Makine Öğrenmesi (Random Forest) ile Getiri Tahminleri Yapılıyor...\n")

ml_beklenen_getiriler = pd.Series(dtype=float)

for hisse in hisseler:
    df = pd.DataFrame(veriler[hisse].rename('Fiyat'))
    df['Getiri'] = df['Fiyat'].pct_change()
    df['Hareketli_Ort_10'] = df['Fiyat'].rolling(window=10).mean()
    df['Hareketli_Ort_30'] = df['Fiyat'].rolling(window=30).mean()
    df['Volatilite'] = df['Getiri'].rolling(window=10).std()

    df['Hedef_Getiri'] = df['Getiri'].shift(-1)
    df = df.dropna()

    X = df[['Fiyat', 'Hareketli_Ort_10', 'Hareketli_Ort_30', 'Volatilite']]
    y = df['Hedef_Getiri']

    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X, y)

    son_veri = X.iloc[-1:]
    tahmin = model.predict(son_veri)[0]
    ml_beklenen_getiriler[hisse] = tahmin * 252

S = risk_models.sample_cov(veriler)

ef_grafik = EfficientFrontier(ml_beklenen_getiriler, S)

fig, ax = plt.subplots(figsize=(10, 6))
plotting.plot_efficient_frontier(ef_grafik, ax=ax, show_assets=True)

ef_hesap = EfficientFrontier(ml_beklenen_getiriler, S)
weights = ef_hesap.max_sharpe()
cleaned_weights = ef_hesap.clean_weights()
ret, std, _ = ef_hesap.portfolio_performance()

ax.scatter(std, ret, marker="*", s=300, c="r", label="Yapay Zeka (Max Sharpe) Portföyü")
ax.set_title("Verim Eğrisi ve ML Destekli Optimum Portföy", fontsize=14, fontweight='bold')
ax.set_xlabel("Risk (Volatilite)")
ax.set_ylabel("Beklenen Getiri")
ax.legend()
plt.show()

print("🤖 ML MODELİNİN ÖNERDİĞİ YENİ PORTFÖY DAĞILIMI:")
print("-" * 45)
for hisse, oran in cleaned_weights.items():
    if oran > 0:
        print(f"👉 {hisse.replace('.IS', '')}: %{oran*100:.1f}")